# Series与DataFrame

学习目标：创建 Series 和 DataFrame，检查标签与类型，区分选择结果，并完成简单列计算。

前置知识：Python 变量、列表与字典、索引、模块导入、基本算术和数组形状。

运行环境：Python 3.12、pandas 3.0、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的数据，后续单元沿用已导入的 pd 和 np。

## 1 DataFrame

DataFrame 是二维的带标签表格，每列可以有不同的数据类型。pd 是 pandas 的常用导入别名。

已有按列整理的数据时，可以用“列名 → 列数据”的字典创建表格。各列表长度必须相同；index 指定行标签，长度应与行数一致。

下面用一张成绩表完成创建、查看、选择和列计算。

In [1]:
import pandas as pd

students = pd.DataFrame(
    {"name": ["小林", "小陈", "小周"], "score": [80, 90, 85]},
    index=["A", "B", "C"],
)

print(students)  # 三行两列；A、B、C 是行标签，name、score 是列名。
print(students.shape)  # 预期：(3, 2)，行标签不作为额外的数据列。

  name  score
A   小林     80
B   小陈     90
C   小周     85
(3, 2)


## 2 表格属性与数据查看

### 2.1 标签、轴与形状

Series 和 DataFrame 保存数据，Index 保存轴标签。DataFrame 的 index 表示行标签，columns 表示列标签；轴 0 对应行，轴 1 对应列，shape 按此顺序给出长度。

| 名称 | 中文名称／含义 |
| --- | --- |
| Series | 一维的带标签数据结构 |
| DataFrame | 二维的带标签表格 |
| Index | 保存轴标签的对象 |

本节继续查看前面创建的 students。示例标签均唯一，但 pandas 不要求标签天然唯一。

In [2]:
print(students.index)  # 行标签依次为 A、B、C。
print(students.columns)  # 列标签依次为 name、score。
print(students.ndim, students.shape)  # 预期：2 (3, 2)。

Index(['A', 'B', 'C'], dtype='str')
Index(['name', 'score'], dtype='str')
2 (3, 2)


### 2.2 列的数据类型

Series 的 dtype 表示数据类型；DataFrame 的 dtypes 返回一个 Series，列名作为标签，列类型作为对应值。

pandas 3 默认把本例字符串列推断为 str，较早版本的示例可能显示 object。下面继续查看 students 的各列类型。

In [3]:
print(students.dtypes)
# name 为 str，score 为 int64。
# 末尾 dtype: object 描述这份类型清单，不是原表全部列的类型。

name       str
score    int64
dtype: object


### 2.3 首尾记录

head(n) 返回前 n 行，tail(n) 返回后 n 行，n 为行数，默认均为 5。它们按当前行顺序取数据，不进行排序。

继续使用 students，分别查看前两行和后两行。

In [4]:
print(students.head(2))  # 前两行：A、B。
print(students.tail(2))  # 后两行：B、C。

  name  score
A   小林     80
B   小陈     90
  name  score
B   小陈     90
C   小周     85


### 2.4 表格摘要

info() 直接输出索引、列名、非缺失数量、类型和内存使用等摘要，返回 None，无需再套 print()。

非缺失数量表示有多少值不是缺失标记，不代表这些值在业务上一定正确。下面查看同一张 students。

In [5]:
students.info()
# 3 条记录、2 列，每列均为 3 non-null。
# name 为 str，score 为 int64；内存显示以实际环境为准。

<class 'pandas.DataFrame'>
Index: 3 entries, A to C
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    3 non-null      str  
 1   score   3 non-null      int64
dtypes: int64(1), str(1)
memory usage: 93.0 bytes


## 3 列与标量选择

### 3.1 单列与多列

列名唯一时，方括号中放一个列名得到 Series，放列名列表得到 DataFrame，即使列表中只有一个列名也一样。选择多个列时，列表决定结果的列顺序。

下面继续使用 students，对比单列与单列表格。

In [6]:
score_column = students["score"]
score_table = students[["score"]]

print(type(score_column), score_column.shape)  # Series，形状为 (3,)。
print(type(score_table), score_table.shape)  # DataFrame，形状为 (3, 1)。
print(score_column.dtype)  # 预期：int64。
print(students[["score", "name"]])  # 三行两列，score 在 name 前面。

<class 'pandas.Series'> (3,)
<class 'pandas.DataFrame'> (3, 1)
int64
   score name
A     80   小林
B     90   小陈
C     85   小周


### 3.2 标量

标量是单个值。用唯一的字符串标签读取 Series，可以得到一个标量，而不是一列或一张表。

下面从刚得到的 score_column 读取 B 对应的成绩。数值标量可以由 NumPy 数值类型表示，仍可直接参与算术运算。

In [7]:
value = score_column["B"]

print(value)  # 预期：90。
print(type(value))  # 预期：numpy.int64。
print(value + 5)  # 预期：95。

90
<class 'numpy.int64'>
95


## 4 列计算

对数值 Series 加上标量，会给每个值加上该标量，并保留标签。把结果赋给 DataFrame 的新列名，可以添加一列；比较运算也能生成布尔列。

下面为 students 新增两列，保留原成绩。计算结果取自同一张表，行标签与原表一致，新列按这些标签放回。

In [8]:
students["adjusted"] = students["score"] + 5
students["at_least_90"] = students["adjusted"] >= 90

print(students)
# 原 name、score 列不变；adjusted 为 85、95、90，at_least_90 为 False、True、True。
print(students.shape)  # 预期：(3, 4)，行数不变，新增两列。
print(students["at_least_90"].dtype)  # 预期：bool。

  name  score  adjusted  at_least_90
A   小林     80        85        False
B   小陈     90        95         True
C   小周     85        90         True
(3, 4)
bool


## 5 其他数据输入

### 5.1 Series 的构造

已有独立的一列数据时，可以直接创建 Series，无需先建表再取列。Series 是一维的带标签数据结构，用一个 dtype 描述数据类型。

从列表创建时，index 指定各值的标签，name 指定 Series 的名称。标签用于标识数据，不增加数据的个数。

In [9]:
scores = pd.Series([80, 90, 85], index=["A", "B", "C"], name="score")

print(scores)  # A、B、C 分别对应 80、90、85，名称为 score。
print(scores.shape)  # 预期：(3,)，一维数据共有三个值。
print(scores.dtype)  # 预期：int64。

A    80
B    90
C    85
Name: score, dtype: int64
(3,)
int64


从列表创建且未指定 index 时，默认生成从 0 开始的整数标签。

如果已有“标签 → 数值”的字典，可以直接传入 Series。未另传 index 时，键成为标签，值成为数据，保留字典的插入顺序。

In [10]:
default_scores = pd.Series([80, 90, 85])
mapped_scores = pd.Series({"B": 90, "A": 80}, name="score")

print(default_scores)  # 标签依次为 0、1、2。
print(mapped_scores)  # 标签依次为 B、A，对应 90、80。

0    80
1    90
2    85
dtype: int64
B    90
A    80
Name: score, dtype: int64


### 5.2 列表输入

数据已按记录排列、每行字段顺序固定时，可以使用嵌套列表。每个内部列表表示一行，columns 按位置指定列名。未指定 index 时使用默认整数行标签。

In [11]:
rows = [["小林", 80], ["小陈", 90]]
from_rows = pd.DataFrame(rows, columns=["name", "score"])

print(from_rows)  # 两行两列，行标签为 0、1，列依次为 name、score。

  name  score
0   小林     80
1   小陈     90


### 5.3 字典记录

每条记录已带有字段名时，可以传入字典的列表。每个字典是一行，键是字段名，值是该行对应的数据；下面的记录均包含相同字段。

In [12]:
records = [{"name": "小林", "score": 80}, {"name": "小陈", "score": 90}]
from_records = pd.DataFrame(records)

print(from_records)  # 两行，列名为 name、score。

  name  score
0   小林     80
1   小陈     90


### 5.4 二维数组

数值数据已保存在二维 NumPy 数组中时，可以直接用它创建 DataFrame，index 和 columns 分别指定行列标签。数组的第一个轴对应行，第二个轴对应列。

下面两列分别表示摄氏温度与相对湿度百分数。

In [13]:
import numpy as np

values = np.array([[20.0, 45.0], [21.5, 50.0]])
from_array = pd.DataFrame(
    values, index=["A", "B"], columns=["temperature_c", "humidity_pct"]
)

print(from_array)  # A 行为 20.0 °C、45.0%；B 行为 21.5 °C、50.0%。
print(from_array.shape)  # 预期：(2, 2)。

   temperature_c  humidity_pct
A           20.0          45.0
B           21.5          50.0
(2, 2)


## 6 综合应用：整理测量数据

创建三条模拟温度记录，检查标签和类型，再按统一的 0.5 °C 修正量生成新列，判断修正后温度是否达到 22.0 °C。

行标签表示记录编号，sensor 表示传感器名称，temperature_c 表示摄氏温度。修正量为教学输入。

In [14]:
# 1. 创建测量表，检查原始结构。
measurements = pd.DataFrame(
    {"sensor": ["S1", "S2", "S3"], "temperature_c": [20.0, 21.5, 23.0]},
    index=["R1", "R2", "R3"],
)
print(measurements.shape)  # 预期：(3, 2)。
print(measurements.dtypes)  # sensor 为 str，temperature_c 为 float64。

# 2. 计算新列，保留原温度。
measurements["adjusted_c"] = measurements["temperature_c"] + 0.5
measurements["at_least_22"] = measurements["adjusted_c"] >= 22.0
print(measurements)
# 行标签仍为 R1、R2、R3；修正后为 20.5、22.0、23.5 °C。
# 布尔结果为 False、True、True；R2 恰好达到阈值。
print(measurements.shape)  # 预期：(3, 4)。

(3, 2)
sensor               str
temperature_c    float64
dtype: object
   sensor  temperature_c  adjusted_c  at_least_22
R1     S1           20.0        20.5        False
R2     S2           21.5        22.0         True
R3     S3           23.0        23.5         True
(3, 4)


## 本章小结

（1）Series 表示一维带标签数据，DataFrame 表示二维表格，Index 保存轴标签。

（2）DataFrame 可以按列或按行构造，也可以接收二维数组；行列标签应与数据形状对应。

（3）shape 表示行列数量，dtypes 表示各列类型；head()、tail() 和 info() 用于查看数据。

（4）列名唯一时，一个列名得到 Series，列名列表得到 DataFrame；Series 按唯一标签取单值可得到标量。

（5）已有列的算术与比较结果可以组成新列。应能判断选择与计算后的结果类型、标签和形状。

## 练习

（1）用下面的数据创建行标签为 A、B、C 的表格，查看前两行、最后一行、形状和各列类型。

In [15]:
data = {"sensor": ["S1", "S2", "S3"], "temperature_c": [18.0, 20.0, 22.0]}
labels = ["A", "B", "C"]

# 在此创建表格并查看数据。
# 检查：shape 为 (3, 2)，行标签不增加数据列；首两行为 A、B，末行为 C。
# 检查：sensor 为 str，temperature_c 为 float64。

（2）先预测下面三个结果的类型，以及前两个结果的形状，再运行核对。

随后完成一个有约束的任务：交付只含 temperature_c 列的数据，必须保留二维表格结构和原行标签。独立写出选择表达式，在代码注释中解释理由，并说明另一种单列写法为什么不符合要求。

In [16]:
sample = pd.DataFrame({"temperature_c": [18.0, 20.0, 22.0]}, index=["A", "B", "C"])
column = sample["temperature_c"]
one_column_table = sample[["temperature_c"]]
value = column["B"]

# 先记录预测，再核对维数以及是否保留行列标签。
print(type(column), column.shape)
print(type(one_column_table), one_column_table.shape)
print(type(value), value)

# 在此按“保留二维表格”的要求重新写出选择表达式，并说明理由。
# 检查：结果为 DataFrame，shape 为 (3, 1)，行标签仍为 A、B、C。
# 数据应与原温度列一致；另一个单列选择结果与本任务要求有何区别？

<class 'pandas.Series'> (3,)
<class 'pandas.DataFrame'> (3, 1)
<class 'numpy.float64'> 20.0


（3）在下面的表中新增 adjusted_c 列，值为原温度加 1.0 °C；再新增布尔列，判断修正后温度是否达到 21.0 °C。保留原温度列。

In [17]:
readings = pd.DataFrame(
    {"sensor": ["S1", "S2", "S3"], "temperature_c": [18.0, 20.0, 22.0]},
    index=["A", "B", "C"],
)

# 在此计算并添加两列，用 print() 检查结果。
# 检查：最终 shape 为 (3, 4)，行标签不变，原温度列保持原值。
# 修正后温度为 19.0、21.0、23.0，布尔结果为 False、True、True。

（4）用下面的二维数组创建表格，行标签为 A、B，列名为 temperature_c、humidity_pct，分别表示摄氏温度和相对湿度百分数。

In [18]:
values = np.array([[18.0, 40.0], [20.0, 45.0]])

# 在此指定 index 和 columns，创建并打印表格。
# 检查：形状为 (2, 2)；A 行为 18.0 °C、40.0%，B 行为 20.0 °C、45.0%。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | 结构与创建：[Intro to data structures](https://pandas.pydata.org/docs/user_guide/dsintro.html) 的 Series、DataFrame、From dict of ndarrays / lists、From a list of dicts、Column selection, addition, deletion；[Series](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) 的 data、index、name、dtype 与标签条件；[DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) 的数据输入和 index、columns。 标签与类型：[Indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html) 的 Basics、Index objects；[DataFrame.shape](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html) 的行列数量；[DataFrame.dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html) 的返回值；[String dtype migration](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#brief-introduction-to-the-new-default-string-dtype) 的默认字符串推断。 查看与计算：[head](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)、[tail](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.tail.html) 的按位置返回首尾行；[info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) 的输出内容与 None 返回值；[Intro to data structures — Vectorized operations and label alignment with Series](https://pandas.pydata.org/docs/user_guide/dsintro.html#vectorized-operations-and-label-alignment-with-series) 的标量运算及标签保留。 |
| NumPy 官方文档（NumPy 2.5） | [array](https://numpy.org/doc/2.5/reference/generated/numpy.array.html) 的嵌套序列创建；[ndarray](https://numpy.org/doc/2.5/reference/arrays.ndarray.html) 的形状与元素类型。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[dsintro](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/dsintro.rst)、[indexing](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/indexing.rst)、[migration-3-strings](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/migration-3-strings.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |